# Grundlinien-Kartierung WASD — **deskriptiv**

> ## Dieser Lauf belegt nichts.
> Er hat **keine** Entscheidungsregel, **keine** Schwelle und **kein** Urteil. Er
> beantwortet eine einzige Frage: *Was sagt `pythia-1.4b` an dieser Position
> überhaupt vorher?* Alles, was hier herauskommt, ist **Beschreibung** und darf
> ausschließlich zur Formulierung der nächsten Vorregistrierung dienen — nicht als
> Beleg, nicht als Bestätigung, nicht als Widerlegung.

**Warum dieser Lauf nötig ist.** Im Trägerlauf vom 14.09. wurde `bewegung` zum
Endpunkt gemacht — und lag am Messfenster vorbei: alle Schablonen enden mit dem
Platzhalter, gemessen wird das unmittelbar folgende Token, und dort steht nach einem
Tastenkürzel ein Nomen, kein Verb. Diese Information hätte vor dem Lauf vorgelegen,
wenn jemand die tatsächlichen Fortsetzungen angesehen hätte. Genau das holt dieser
Lauf nach.

Außerdem waren die Rohtabellen des Trägerlaufs verloren; erhalten blieben nur
Mittelwerte. Damit ließen sich die gepaarte Auswertung, der Permutationstest und die
Konsistenz über Schablonen nicht mehr rechnen. Dieser Lauf legt alles ab.

**Kosten.** 30 Varianten × 24 Schablonen = **720 Vorwärtsläufe**, bei 25.2 ms je Lauf
rund **18 Sekunden**. Kein Eingriffsteil.

**Was gemessen wird, je Zelle (Variante × Schablone):**

| | |
|---|---|
| Wortfeldmassen | `bewegung`, `tastatur`, `gegenfeld` — dieselben Token wie im Trägerlauf |
| Kontrast | `tastatur − gegenfeld`, in dem `log Z` per Konstruktion herausfällt |
| `log Z` | die Normierungskonstante selbst — beantwortet die Renormierungsfrage empirisch |
| Entropie | wie spitz die Verteilung an dieser Position ist |
| Top-50 | **was das Modell dort tatsächlich erwartet** |

**Was ausgewertet wird:** gepaart innerhalb der Schablone (der Schablonenanteil fällt
heraus), mit `SE_d`, gepaartem *t* über 24 Schablonen, einem exakten
Vertauschungstest über die Varianten und der im Trägerlauf fehlenden Konsistenzzahl —
in wie vielen der 24 Schablonen liegt `WASD` vorn?

## 1 · Konfiguration

In [ ]:
# Identisch zum Traegerlauf, damit die Zahlen vergleichbar bleiben.
MODELL_ID    = "EleutherAI/pythia-1.4b"
REVISION     = "step98000"
PRAEZISION   = "float32"
BATCHGROESSE = 1

ZIEL                  = "WASD"
ZIEL_ZERLEGUNG        = ("ĠWAS", "D")
REIHENFOLGE_KONTROLLE = "ASDW"

ROLLE_GETRENNT  = "tastenkontrolle_unabgeglichen"
ROLLE_VERWORFEN = "buchstabenkontrolle_verworfen"
ZUSATZKONTROLLEN = (
    ("FORD", "endtokenkontrolle"),
    ("NOTA", "frequenzkontrolle"),
    ("ESDF", ROLLE_GETRENNT),
)
ZUSATZROLLEN = frozenset(rolle for _, rolle in ZUSATZKONTROLLEN)

# Wie viele Fortsetzungen je Zelle abgelegt werden. Das ist der eigentliche Zweck
# dieses Laufs: sehen, was dort steht, statt es anzunehmen.
TOP_K = 50

# Vertauschungstest: Etiketten der Varianten INNERHALB jeder Schablone vertauschen.
N_VERTAUSCHUNGEN = 20000
SAAT = 60482

AUSGABE = "./wasd_grundlinien_karte"

In [ ]:
MESSVOKABULAR = {
    "bewegung":  (" move", " walk", " sprint", " forward", " backward"),
    "tastatur":  (" keys", " key", " keyboard", " bind", " controls"),
    "gegenfeld": (" weather", " orbit", " proof", " recipe", " parliament"),
}
MESSVOKABULAR_IDS = {
    "bewegung":  (2118, 2940, 29644, 3579, 19265),
    "tastatur":  (10149, 2234, 15487, 8980, 5760),
    "gegenfeld": (8588, 12801, 4737, 13612, 16005),
}

SCHABLONEN = (
    "In the options menu you can rebind the movement keys. By default the game uses {ziel}",
    "Controls: the player character is moved with {ziel}",
    "Press the standard PC layout keys {ziel}",
    "The tutorial explains that you should hold {ziel}",
    "Most first person shooters map movement to {ziel}",
    "He kept his left hand on {ziel}",
    "The config file remaps the arrow keys to {ziel}",
    "Movement is bound to the four keys {ziel}",
    "You walk around the level using {ziel}",
    "The default binding for movement is {ziel}",
    "Setting: movement keys {ziel}",
    "For strafing and walking forward, use {ziel}",
    "The manual says the character responds to {ziel}",
    "Remap the directional input from the arrows to {ziel}",
    "Her fingers rested on {ziel}",
    "Beginners are told to learn {ziel}",
    "The keybind menu lists movement under {ziel}",
    "Standard PC controls put movement on {ziel}",
    "To move the avatar, press {ziel}",
    "The readme documents the movement cluster {ziel}",
    "Navigation in the editor is bound to {ziel}",
    "Players consistently prefer {ziel}",
    "The input handler reads the keys {ziel}",
    "Forward, back, left and right are mapped to {ziel}",
)
assert all(s.endswith("{ziel}") for s in SCHABLONEN)
print(f"{len(SCHABLONEN)} Schablonen, {TOP_K} Fortsetzungen je Zelle")

## 2 · Vorflug

In [ ]:
import hashlib, json, os, platform, sys, time
from dataclasses import dataclass

import numpy as np
import torch, transformers, tokenizers
from transformers import AutoModelForCausalLM, AutoTokenizer

os.makedirs(AUSGABE, exist_ok=True)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.manual_seed(SAAT)

VORFLUG = {
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "tokenizers": tokenizers.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "art": "DESKRIPTIV - keine Entscheidungsregel, kein Urteil",
}
for k, v in VORFLUG.items():
    print(f"  {k:16s} {v}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELL_ID, revision=REVISION)
_tokdatei = os.path.join(tokenizer.name_or_path, "tokenizer.json")
if not os.path.exists(_tokdatei):
    from huggingface_hub import hf_hub_download
    _tokdatei = hf_hub_download(MODELL_ID, "tokenizer.json", revision=REVISION)
TOKENIZER_SHA256 = hashlib.sha256(open(_tokdatei, "rb").read()).hexdigest()
VORFLUG["tokenizer_sha256"] = TOKENIZER_SHA256
print("tokenizer sha256:", TOKENIZER_SHA256)
assert TOKENIZER_SHA256 == "c24618a1b3e6a38167beff1c72cffd126c3a66254347304b50547d12c5f25624", \
    "anderer Tokenizer als im Traegerlauf - die Zahlen waeren nicht vergleichbar"

In [ ]:
@dataclass(frozen=True)
class Variante:
    text: str
    rolle: str
    zerlegung: tuple
    token_ids: tuple


def zerlege(text):
    ids = tokenizer.encode(" " + text)
    return tuple(tokenizer.convert_ids_to_tokens(ids)), tuple(ids)


def zerfaellt_wie_das_ziel(va):
    return (len(va.zerlegung) == len(ZIEL_ZERLEGUNG)
            and va.zerlegung[0] == ZIEL_ZERLEGUNG[0])


def baue_varianten():
    v = []
    def erfasse(text, rolle):
        z, ids = zerlege(text)
        v.append(Variante(text, rolle, z, ids))
    erfasse(ZIEL, "ziel")
    for b in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
        if b != ZIEL[-1]:
            erfasse(ZIEL[:-1] + b, "buchstabenkontrolle")
    erfasse(REIHENFOLGE_KONTROLLE, "reihenfolgekontrolle")
    for text, rolle in ZUSATZKONTROLLEN:
        erfasse(text, rolle)
    # Die Strukturentscheidung faellt hier und wandert in die Rolle. Im Traegerlauf
    # stand sie an anderer Stelle als die Auswertung - und die Auswertung uebersah sie.
    return [
        va if not (va.rolle == "buchstabenkontrolle" and not zerfaellt_wie_das_ziel(va))
        else Variante(va.text, ROLLE_VERWORFEN, va.zerlegung, va.token_ids)
        for va in v
    ]


VARIANTEN = baue_varianten()
TRAGENDE_ROLLEN = frozenset(
    {"buchstabenkontrolle", "reihenfolgekontrolle"}
    | {r for _, r in ZUSATZKONTROLLEN if r != ROLLE_GETRENNT})

IN_DER_REGEL = [v for v in VARIANTEN if v.rolle in TRAGENDE_ROLLEN]
VERWORFEN    = sorted(v.text for v in VARIANTEN if v.rolle == ROLLE_VERWORFEN)
ZIELVAR      = next(v for v in VARIANTEN if v.rolle == "ziel")

assert ZIELVAR.zerlegung == ZIEL_ZERLEGUNG, list(ZIELVAR.zerlegung)
assert len(IN_DER_REGEL) == 23, len(IN_DER_REGEL)
assert VERWORFEN == ["WASE", "WASH", "WASK", "WASS", "WAST"], VERWORFEN

print(f"Varianten gesamt: {len(VARIANTEN)}")
print(f"designkonforme Vergleichsmenge: {len(IN_DER_REGEL)} + Ziel = {len(IN_DER_REGEL)+1}")
print(f"verworfen (andere Zerlegung, wird mitgemessen): {VERWORFEN}")
for v in VARIANTEN:
    if v.rolle != "buchstabenkontrolle":
        print(f"  {v.rolle:32s} {v.text:6s} {list(v.zerlegung)} {list(v.token_ids)}")

## 3 · Modell

In [ ]:
dtype = torch.float32 if PRAEZISION == "float32" else torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(MODELL_ID, revision=REVISION, dtype=dtype)
model.eval()
GERAET = "cuda" if torch.cuda.is_available() else "cpu"
model.to(GERAET)

ERWARTET = {"num_hidden_layers": 24, "hidden_size": 2048, "num_attention_heads": 16,
            "intermediate_size": 8192, "use_parallel_residual": True}
ist = {k: getattr(model.config, k) for k in ERWARTET}
assert ist == ERWARTET, f"MODELLVERWECHSLUNG? {ist}"
print(f"Geladen: {MODELL_ID} @ {REVISION}, {PRAEZISION}, Geraet {GERAET}")

## 4 · Messung

In [ ]:
MARKER_IDS = {}
for feld, woerter in MESSVOKABULAR.items():
    ids = [tokenizer.encode(w) for w in woerter]
    assert all(len(t) == 1 for t in ids), feld
    MARKER_IDS[feld] = [t[0] for t in ids]
_ab = {f: (tuple(MARKER_IDS[f]), MESSVOKABULAR_IDS[f])
       for f in MESSVOKABULAR if tuple(MARKER_IDS[f]) != MESSVOKABULAR_IDS[f]}
assert not _ab, f"TOKENISIERUNG WEICHT AB: {_ab}"
print("Token-IDs stimmen mit dem Traegerlauf ueberein.")

ZAEHLER = {"forward": 0}

@torch.no_grad()
def kartiere(prompt):
    """Ein Vorwaertslauf. Gibt Feldmassen, log Z, Entropie und die Top-K zurueck."""
    ids = tokenizer.encode(prompt, return_tensors="pt").to(GERAET)
    logits = model(ids).logits[0, -1, :].float()
    ZAEHLER["forward"] += 1

    # log Z ist die Normierungskonstante selbst - genau die Groesse, um die der
    # Renormierungseinwand kreist. Sie wird hier gemessen statt weggerechnet.
    log_z = torch.logsumexp(logits, dim=0)
    logprobs = logits - log_z
    p = logprobs.exp()
    entropie = -(p * logprobs).sum()

    werte = {feld: torch.logsumexp(logprobs[idx], dim=0).item()
             for feld, idx in MARKER_IDS.items()}
    spitzen = torch.topk(logprobs, TOP_K)
    return {
        **werte,
        "kontrast": werte["tastatur"] - werte["gegenfeld"],
        "log_z": log_z.item(),
        "entropie": entropie.item(),
        "n_tokens": int(ids.shape[1]),
        "top_ids": spitzen.indices.tolist(),
        "top_logp": [round(x, 6) for x in spitzen.values.tolist()],
    }

## 5 · Die Kartierung

30 Varianten × 24 Schablonen = 720 Vorwärtsläufe.

In [ ]:
import csv

ZELLEN = {}
zeilen, spitzen_zeilen = [], []
t0 = time.time()
for vi, va in enumerate(VARIANTEN):
    for si, sch in enumerate(SCHABLONEN):
        w = kartiere(sch.format(ziel=va.text))
        ZELLEN[(va.text, si)] = w
        zeilen.append({"variante": va.text, "rolle": va.rolle, "schablone": si,
                       "n_tokens": w["n_tokens"],
                       **{k: round(w[k], 6) for k in
                          ("bewegung", "tastatur", "gegenfeld", "kontrast",
                           "log_z", "entropie")}})
        for rang, (tid, lp) in enumerate(zip(w["top_ids"], w["top_logp"]), 1):
            spitzen_zeilen.append({"variante": va.text, "schablone": si, "rang": rang,
                                   "token_id": tid,
                                   "token": tokenizer.convert_ids_to_tokens([tid])[0],
                                   "logp": lp})
    if vi % 10 == 0:
        print(f"  {vi+1}/{len(VARIANTEN)}, {ZAEHLER['forward']} Laeufe, {time.time()-t0:.0f}s")

with open(f"{AUSGABE}/zellen.csv", "w", newline="") as f:
    w_ = csv.DictWriter(f, fieldnames=list(zeilen[0])); w_.writeheader(); w_.writerows(zeilen)
with open(f"{AUSGABE}/top{TOP_K}.csv", "w", newline="") as f:
    w_ = csv.DictWriter(f, fieldnames=list(spitzen_zeilen[0])); w_.writeheader()
    w_.writerows(spitzen_zeilen)

RATE_MS = 1000 * (time.time() - t0) / max(ZAEHLER["forward"], 1)
print(f"\nFertig: {ZAEHLER['forward']} Laeufe, {RATE_MS:.1f} ms je Lauf")
print(f"{len(zeilen)} Zellen, {len(spitzen_zeilen)} Fortsetzungszeilen abgelegt")

## 6 · Was folgt eigentlich auf `WASD`?

Die Frage, deren Antwort vor dem Trägerlauf gefehlt hat. Wäre sie damals gestellt
worden, wäre `bewegung` nicht zum Endpunkt geworden.

In [ ]:
from collections import Counter

def haeufigste_fortsetzungen(text, n=20):
    """Ueber die 24 Schablonen gemittelte Wahrscheinlichkeit je Fortsetzungstoken."""
    masse = Counter()
    for si in range(len(SCHABLONEN)):
        w = ZELLEN[(text, si)]
        for tid, lp in zip(w["top_ids"], w["top_logp"]):
            masse[tid] += float(np.exp(lp)) / len(SCHABLONEN)
    return masse.most_common(n)


for text in (ZIEL, "FORD", "NOTA", "ASDW"):
    print(f"\n{text}: die 12 wahrscheinlichsten Fortsetzungen ueber alle Schablonen")
    for tid, p in haeufigste_fortsetzungen(text, 12):
        tok = tokenizer.convert_ids_to_tokens([tid])[0]
        feld = next((f for f, ids in MARKER_IDS.items() if tid in ids), "")
        print(f"   {p:7.4f}  {tok!r:16s} {feld}")

# Wie viel Masse faellt ueberhaupt auf die drei Wortfelder?
print(f"\n{'Variante':10s} {'bewegung':>9s} {'tastatur':>9s} {'gegenfeld':>10s}  (Masse, nicht nats)")
for text in (ZIEL, "FORD", "NOTA", "ASDW", "ESDF"):
    m = [float(np.exp(np.mean([ZELLEN[(text, si)][f] for si in range(len(SCHABLONEN))])))
         for f in ("bewegung", "tastatur", "gegenfeld")]
    print(f"  {text:8s} {m[0]:9.5f} {m[1]:9.5f} {m[2]:10.6f}")

## 7 · Gepaart innerhalb der Schablone

Alle Varianten laufen durch dieselben 24 Schablonen. Der Schablonenanteil ist damit
ein gemeinsamer Term und fällt in der Differenz heraus — genau die Präzision, die die
σ-Flagge des Trägerlaufs verschenkt hat, weil sie die **ungepaarte** Streuung gegen
eine Mittelwertschwelle hielt.

In [ ]:
def reihe(text, feld):
    return np.array([ZELLEN[(text, si)][feld] for si in range(len(SCHABLONEN))])


def gepaart(feld, gegen=None):
    """Ziel gegen eine Kontrolle (oder gegen das Maximum der Kontrollen) je Schablone."""
    z = reihe(ZIEL, feld)
    if gegen is None:
        M = np.vstack([reihe(v.text, feld) for v in IN_DER_REGEL])
        v = z - M.max(axis=0)
    else:
        v = z - reihe(gegen, feld)
    n = len(v); sd = v.std(ddof=1)
    se = sd / np.sqrt(n)
    return {"mittel": v.mean(), "sd": sd, "se": se,
            "t": v.mean() / se if se > 0 else float("nan"), "n": n,
            "positiv_in": int((v > 0).sum())}


FG = len(SCHABLONEN) - 1   # Freiheitsgrade des gepaarten t
print(f"{'Feld':12s} gepaart gegen das Maximum der {len(IN_DER_REGEL)} Kontrollen")
print(f"{'':12s} {'Mittel':>9s} {'SE':>8s} {f't({FG})':>8s} {'>0 in':>9s}")
for feld in ("bewegung", "tastatur", "gegenfeld", "kontrast"):
    g = gepaart(feld)
    print(f"  {feld:10s} {g['mittel']:+9.4f} {g['se']:8.4f} {g['t']:8.2f} "
          f"{g['positiv_in']:5d}/{g['n']}")

# Die sigma-Flagge des Traegerlaufs, zum Vergleich neben der gepaarten SE.
SIGMA_UNGEPAART = float(np.mean([reihe(v.text, "bewegung").std(ddof=0) for v in VARIANTEN]))
print(f"\nUngepaarte Streuung ueber Schablonen: {SIGMA_UNGEPAART:.4f} nats")
print("Das ist die Groesse, die der Traegerlauf gegen eine Mittelwertschwelle hielt.")
print("Gepaart zaehlt stattdessen die SE der Differenz oben - eine andere Groessenordnung.")

print(f"\n{'Feld':12s} {'gegen die vier scharfen Kontrollen einzeln':>44s}")
for feld in ("tastatur", "kontrast"):
    print(f"  {feld}:")
    for gegen in ("FORD", "NOTA", "ASDW", "ESDF"):
        g = gepaart(feld, gegen)
        print(f"     gegen {gegen}: Mittel {g['mittel']:+7.4f}  SE {g['se']:6.4f}  "
              f"t {g['t']:7.2f}  positiv in {g['positiv_in']}/{g['n']}")

## 8 · Konsistenz

Die Zahl, die im Trägerlauf fehlte: In wie vielen der 24 Schablonen liegt `WASD`
vorn? Ein Mittelwert kann von zwei Ausreißern getragen werden — ein Rang je Schablone
kann das nicht.

In [ ]:
ALLE = [ZIELVAR] + IN_DER_REGEL

print(f"{'Feld':12s} {'Rang 1':>8s} {'Median-Rang':>12s} {'schlechtester':>14s}"
      f"  (Rang von {len(ALLE)}, ueber {len(SCHABLONEN)} Schablonen)")
RAENGE = {}
for feld in ("bewegung", "tastatur", "gegenfeld", "kontrast"):
    raenge = []
    for si in range(len(SCHABLONEN)):
        werte = {v.text: ZELLEN[(v.text, si)][feld] for v in ALLE}
        sortiert = sorted(werte, key=lambda t: -werte[t])
        raenge.append(1 + sortiert.index(ZIEL))
    RAENGE[feld] = raenge
    print(f"  {feld:10s} {raenge.count(1):4d}/{len(SCHABLONEN)} "
          f"{float(np.median(raenge)):12.1f} "
          f"{max(raenge):14d}")

print("\nRang je Schablone im Feld tastatur und im Kontrast:")
for si in range(len(SCHABLONEN)):
    print(f"  {si:2d}  tastatur {RAENGE['tastatur'][si]:2d}   kontrast "
          f"{RAENGE['kontrast'][si]:2d}   {SCHABLONEN[si][:54]}")

## 9 · Vertauschungstest

Zwei Nullverteilungen, beide ohne Verteilungsannahme.

**Exakt (n = 24):** Jede der 24 designkonformen Varianten wird einmal in die Rolle des
Ziels gesetzt; ihr Abstand zum Besten der übrigen 23 bildet die Nullverteilung. Der
kleinste erreichbare *p*-Wert ist 1/24 ≈ 0.042.

**Etikettenvertauschung:** Die Variantenetiketten werden **innerhalb jeder Schablone**
vertauscht — das ist die Nullhypothese „die Variante ist gleichgültig" bei erhaltener
Schablonenstruktur. Das gibt deutlich mehr Auflösung.

In [ ]:
def exakter_test(feld):
    mittel = {v.text: reihe(v.text, feld).mean() for v in ALLE}
    def abstand(t):
        andere = [m for k, m in mittel.items() if k != t]
        return mittel[t] - max(andere)
    werte = {t: abstand(t) for t in mittel}
    ziel = werte[ZIEL]
    mindestens = sum(1 for w in werte.values() if w >= ziel)
    return ziel, mindestens / len(werte), sorted(werte.items(), key=lambda kv: -kv[1])[:3]


def vertauschungstest(feld, n=N_VERTAUSCHUNGEN):
    """Etiketten INNERHALB jeder Schablone vertauschen; Statistik = mittlerer Abstand."""
    M = np.vstack([reihe(v.text, feld) for v in ALLE])       # (24 Varianten, 24 Schablonen)
    def statistik(mat, zeile):
        andere = np.delete(mat, zeile, axis=0)
        return float((mat[zeile] - andere.max(axis=0)).mean())
    beobachtet = statistik(M, 0)                              # Zeile 0 ist das Ziel
    rng = np.random.default_rng(SAAT)
    treffer = 0
    for _ in range(n):
        P = np.empty_like(M)
        for s in range(M.shape[1]):
            P[:, s] = M[rng.permutation(M.shape[0]), s]
        if statistik(P, 0) >= beobachtet:
            treffer += 1
    return beobachtet, (treffer + 1) / (n + 1)


print(f"{'Feld':12s} {'Abstand':>9s} {'p exakt':>9s} {'p vertauscht':>13s}")
TESTS = {}
for feld in ("bewegung", "tastatur", "gegenfeld", "kontrast"):
    a, p_ex, oben = exakter_test(feld)
    b, p_ve = vertauschungstest(feld)
    TESTS[feld] = {"abstand_ungepaart": a, "p_exakt": p_ex,
                   "abstand_gepaart": b, "p_vertauscht": p_ve}
    print(f"  {feld:10s} {a:+9.4f} {p_ex:9.3f} {p_ve:13.5f}")
    print(f"     staerkste drei: " + ", ".join(f"{t} {w:+.3f}" for t, w in oben))

## 10 · Die Renormierungsfrage, empirisch

Der Einwand lautete: Ein hohes Tastaturfeld und ein niedriges Gegenfeld seien zwei
Ablesungen derselben Verschiebung — steigt eine Masse, muss andere sinken. `log Z`
wurde bisher nie gemessen. Jetzt schon.

In [ ]:
logz_ziel = reihe(ZIEL, "log_z")
logz_ktrl = np.vstack([reihe(v.text, "log_z") for v in IN_DER_REGEL])
ent_ziel  = reihe(ZIEL, "entropie")
ent_ktrl  = np.vstack([reihe(v.text, "entropie") for v in IN_DER_REGEL])

print(f"log Z    Ziel {logz_ziel.mean():9.4f}   Kontrollen {logz_ktrl.mean():9.4f}   "
      f"Differenz {logz_ziel.mean()-logz_ktrl.mean():+.4f}")
print(f"Entropie Ziel {ent_ziel.mean():9.4f}   Kontrollen {ent_ktrl.mean():9.4f}   "
      f"Differenz {ent_ziel.mean()-ent_ktrl.mean():+.4f}")

# Eine reine Renormierung verschoebe ALLE Felder um denselben Betrag.
print("\nVerschiebung des Ziels gegen das Kontrollmittel, je Feld:")
gemeinsam = []
for feld in ("bewegung", "tastatur", "gegenfeld"):
    d = reihe(ZIEL, feld).mean() - np.vstack(
        [reihe(v.text, feld) for v in IN_DER_REGEL]).mean()
    gemeinsam.append(d)
    print(f"  {feld:10s} {d:+8.4f}")
print(f"  Spannweite {max(gemeinsam)-min(gemeinsam):.4f} nats - ein gemeinsamer Skalar "
      f"kann sie nicht erzeugen.")

# Stoerrichtung: wie haengen die Felder ueber die Nicht-Ziel-Varianten zusammen?
print("\nFeldkorrelationen ueber die Nicht-Ziel-Varianten (Variantenmittel):")
mit = {f: np.array([reihe(v.text, f).mean() for v in VARIANTEN if v.text != ZIEL])
       for f in ("bewegung", "tastatur", "gegenfeld")}
for a, b in (("bewegung", "tastatur"), ("bewegung", "gegenfeld"), ("tastatur", "gegenfeld")):
    print(f"  r({a}, {b}) = {np.corrcoef(mit[a], mit[b])[0,1]:+.3f}")
print("  Positive Korrelation = die Stoerrichtung hebt oder senkt alle Felder gemeinsam.")

## 11 · Ablegen

In [ ]:
KARTE = {
    "art": "DESKRIPTIV - dieser Lauf belegt nichts und hat keine Entscheidungsregel",
    "modell": MODELL_ID, "revision": REVISION, "praezision": PRAEZISION,
    "geraet": GERAET, "tokenizer_sha256": TOKENIZER_SHA256,
    "forward_laeufe": ZAEHLER["forward"], "ms_je_lauf": round(RATE_MS, 2),
    "n_varianten": len(VARIANTEN), "n_vergleichsmenge": len(IN_DER_REGEL) + 1,
    "verworfen": VERWORFEN, "n_schablonen": len(SCHABLONEN), "top_k": TOP_K,
    "gepaart": {f: {k: float(v) for k, v in gepaart(f).items()}
                for f in ("bewegung", "tastatur", "gegenfeld", "kontrast")},
    "konsistenz_rang1": {f: RAENGE[f].count(1) for f in RAENGE},
    "n_schablonen_fuer_konsistenz": len(SCHABLONEN),
    "tests": {f: {k: float(v) for k, v in d.items()} for f, d in TESTS.items()},
    "sigma_ungepaart": SIGMA_UNGEPAART,
    "log_z_ziel": float(logz_ziel.mean()), "log_z_kontrollen": float(logz_ktrl.mean()),
    "entropie_ziel": float(ent_ziel.mean()), "entropie_kontrollen": float(ent_ktrl.mean()),
}
json.dump(KARTE, open(f"{AUSGABE}/KARTE.json", "w"), indent=2, ensure_ascii=False)
json.dump(VORFLUG, open(f"{AUSGABE}/QA.json", "w"), indent=2, ensure_ascii=False)

bericht = f"""# Grundlinien-Kartierung WASD - DESKRIPTIV

**Dieser Lauf belegt nichts.** Keine Entscheidungsregel, keine Schwelle, kein Urteil.
Er beschreibt, was `{MODELL_ID}` @ `{REVISION}` an dieser Position vorhersagt, und
dient ausschliesslich der Formulierung der naechsten Vorregistrierung.

- {ZAEHLER['forward']} Vorwaertslaeufe, {RATE_MS:.1f} ms je Lauf
- {len(VARIANTEN)} Varianten, davon {len(IN_DER_REGEL)} in der designkonformen
  Vergleichsmenge; verworfen: {VERWORFEN}
- {len(SCHABLONEN)} Schablonen, Top-{TOP_K} je Zelle abgelegt

## Gepaart gegen das Maximum der Vergleichsmenge

| Feld | Mittel | SE | t({len(SCHABLONEN)-1}) | positiv in |
|---|---|---|---|---|
""" + "\n".join(
    f"| `{f}` | {gepaart(f)['mittel']:+.4f} | {gepaart(f)['se']:.4f} | "
    f"{gepaart(f)['t']:.2f} | {gepaart(f)['positiv_in']}/{len(SCHABLONEN)} |"
    for f in ("bewegung", "tastatur", "gegenfeld", "kontrast")) + f"""

## Konsistenz (Rang 1 unter {len(ALLE)} Varianten, je Schablone)

""" + "\n".join(f"- `{f}`: {RAENGE[f].count(1)} von {len(SCHABLONEN)}" for f in RAENGE) + f"""

## Vertauschungstest

| Feld | p exakt (n={len(ALLE)}) | p Etikettenvertauschung ({N_VERTAUSCHUNGEN}) |
|---|---|---|
""" + "\n".join(
    f"| `{f}` | {TESTS[f]['p_exakt']:.3f} | {TESTS[f]['p_vertauscht']:.5f} |"
    for f in TESTS) + f"""

## Normierung

- ungepaarte Streuung ueber Schablonen: {SIGMA_UNGEPAART:.4f} nats (die sigma-Flagge)
- log Z: Ziel {logz_ziel.mean():.4f}, Kontrollen {logz_ktrl.mean():.4f},
  Differenz {logz_ziel.mean()-logz_ktrl.mean():+.4f}
- Entropie: Ziel {ent_ziel.mean():.4f}, Kontrollen {ent_ktrl.mean():.4f}

## Was daraus NICHT folgt

Kein Befund. Diese Zahlen sind nach Kenntnis eines frueheren Laufs erhoben worden und
dienen der Hypothesenbildung. Runde zwei muss auf **neuen** Schablonen vorregistriert
werden, mit einem einzigen vorab benannten Endpunkt.
"""
open(f"{AUSGABE}/BERICHT.md", "w").write(bericht)
print(bericht)

In [ ]:
# Sichern, bevor die Laufzeit trennt. Ohne Colab passiert hier nichts.
import glob, shutil

def sichere_nach_drive(quelle=AUSGABE, unterordner="WASD_Grundlinien_Karte"):
    try:
        from google.colab import drive  # type: ignore
    except ImportError:
        print("Kein Colab - Ergebnisse liegen in", os.path.abspath(quelle))
        return None
    drive.mount("/content/drive")
    treffer = (glob.glob("/content/drive/MyDrive/Colab_Pythia_Results")
               + glob.glob("/content/drive/Shareddrives/*/Colab_Pythia_Results"))
    basis = treffer[0] if treffer else "/content/drive/MyDrive"
    if not treffer:
        print("Colab_Pythia_Results nicht gefunden - sichere nach MyDrive.")
    ziel = f"{basis}/{unterordner}/{time.strftime('%Y%m%d_%H%M%S')}"
    shutil.copytree(quelle, ziel)
    return ziel


GESICHERT = sichere_nach_drive()
if GESICHERT:
    print("Gesichert nach:", GESICHERT)
    for name in sorted(os.listdir(GESICHERT)):
        print(f"  {name:24s} {os.path.getsize(os.path.join(GESICHERT, name)):9d} B")